# Entrenamiento YOLOv10

Notebook para entrenar el modelo de detección de productos.

## Configuración

In [ ]:
import os
os.environ.pop('MPLBACKEND', None)  # Remove conflicting env var

import matplotlib
matplotlib.use('Agg')

from pathlib import Path
from datetime import datetime

from ultralytics import YOLO

In [ ]:
# Hiperparámetros
MODEL_NAME = "yolov10n.pt"  # Opciones: yolov10n/s/m/b/l/x
DATA_YAML = Path("../data/poc_10_clases/data.yaml").resolve()
EPOCHS = 100
IMGSZ = 640
BATCH = 16
PROJECT = "../runs/train"
NAME = datetime.now().strftime("%Y%m%d_%H%M%S")

# Resolve absolute path for YOLO
print(f"Dataset path: {DATA_YAML}")

## Verificar dataset

In [ ]:
import yaml

# Verificar que existe el archivo data.yaml
assert DATA_YAML.exists(), f"No se encontró: {DATA_YAML}"

# Leer y agregar path absoluto
with open(DATA_YAML) as f:
    data_config = yaml.safe_load(f)

# Agregar path absoluto al dataset
data_config['path'] = str(DATA_YAML.parent)

# Guardar versión con path absoluto
with open(DATA_YAML, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False, sort_keys=False)

print("data.yaml actualizado con path absoluto:")
print(DATA_YAML.read_text())

In [ ]:
# Contar imágenes en train y val
train_images = list((DATA_YAML.parent / "train" / "images").glob("*"))
val_images = list((DATA_YAML.parent / "val" / "images").glob("*"))

print(f"Imágenes de entrenamiento: {len(train_images)}")
print(f"Imágenes de validación: {len(val_images)}")

## Cargar modelo base

In [ ]:
model = YOLO(MODEL_NAME)
print(f"Modelo cargado: {MODEL_NAME}")

## Entrenar

In [ ]:
results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    project=PROJECT,
    name=NAME,
    patience=20,
    save=True,
    plots=True,
    verbose=True,
)

## Resultados

In [ ]:
# Path al mejor modelo
best_model_path = Path(PROJECT) / NAME / "weights" / "best.pt"
print(f"Mejor modelo guardado en: {best_model_path}")

In [ ]:
# Mostrar métricas finales
print("\nMétricas de entrenamiento:")
print(f"  mAP50: {results.results_dict.get('metrics/mAP50(B)', 'N/A')}")
print(f"  mAP50-95: {results.results_dict.get('metrics/mAP50-95(B)', 'N/A')}")
print(f"  Precision: {results.results_dict.get('metrics/precision(B)', 'N/A')}")
print(f"  Recall: {results.results_dict.get('metrics/recall(B)', 'N/A')}")

## Visualizar curvas de entrenamiento

In [ ]:
from IPython.display import Image, display

results_dir = Path(PROJECT) / NAME

# Mostrar curvas si existen
for plot_name in ["results.png", "confusion_matrix.png", "PR_curve.png"]:
    plot_path = results_dir / plot_name
    if plot_path.exists():
        print(f"\n{plot_name}:")
        display(Image(filename=str(plot_path)))

## Validación

In [ ]:
# Cargar el mejor modelo y validar
best_model = YOLO(str(best_model_path))
val_results = best_model.val(data=str(DATA_YAML))

## Prueba rápida de inferencia

In [ ]:
# Probar con una imagen de validación
if val_images:
    test_image = val_images[0]
    print(f"Probando con: {test_image}")
    
    results = best_model.predict(source=str(test_image), conf=0.5)
    
    # Mostrar resultado
    for r in results:
        im_array = r.plot()
        display(Image(data=im_array))

## Copiar modelo a models/

In [ ]:
import shutil

models_dir = Path("../models")
models_dir.mkdir(exist_ok=True)

dest_path = models_dir / "best.pt"
shutil.copy(best_model_path, dest_path)
print(f"Modelo copiado a: {dest_path}")